In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense, Dropout, Bidirectional, Conv1D, Flatten
from tensorflow.keras.callbacks import EarlyStopping
from sqlalchemy import create_engine
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from tqdm.notebook import tqdm
import os
from datetime import datetime
from pathlib import Path

# --- 1. Database & Path Setup ---
db_connection_str = "mysql+pymysql://root:@127.0.0.1/trading_system"
db_engine = create_engine(db_connection_str)

base_dir = Path.cwd().parent
output_dir = base_dir / "4_Results" / "Full_Ablation_Runs"
output_dir.mkdir(parents=True, exist_ok=True)

# --- 2. Define the ISOLATED Feature Sets ---
# Matching your Traditional setup: Testing the 3 specific sources only
feature_variants = {
    "VADER":      ['close', 'volume', 'vader_compound'],
    "TextBlob":   ['close', 'volume', 'textblob_polarity'],
    "FinBERT":    ['close', 'volume', 'finbert_compound'],
}

# --- 3. Model Builders ---
def build_model(model_type, input_shape):
    model = Sequential()
    if model_type == 'LSTM':
        model.add(LSTM(50, input_shape=input_shape, return_sequences=False))
    elif model_type == 'BiLSTM':
        model.add(Bidirectional(LSTM(50, return_sequences=False), input_shape=input_shape))
    elif model_type == 'GRU':
        model.add(GRU(50, input_shape=input_shape, return_sequences=False))
    elif model_type == 'CNN':
        model.add(Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=input_shape))
        model.add(Conv1D(filters=32, kernel_size=3, activation='relu'))
        model.add(Flatten())
    
    model.add(Dropout(0.2))
    model.add(Dense(25, activation='relu'))
    model.add(Dense(1)) 
    model.compile(optimizer='adam', loss='mse')
    return model

# --- 4. Sequence Generator ---
def create_sequences(data, feature_cols, sequence_length=60):
    data = data.sort_values('date')
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(data[feature_cols])
    target = data['close'].pct_change().shift(-1).fillna(0).values
    
    X, y = [], []
    for i in range(sequence_length, len(data) - 1):
        X.append(scaled_data[i-sequence_length:i])
        y.append(target[i])
    return np.array(X), np.array(y)

# --- 5. MASSIVE TRAINING LOOP (Capped at 2000) ---
print("Loading FULL dataset...")
df = pd.read_sql("SELECT * FROM data_daily_merged", con=db_engine)
df['date'] = pd.to_datetime(df['date'])

# --- TICKER SELECTION LOGIC ---
# We prioritize tickers with the MOST data points to ensure DL convergence
print("Selecting Top 2,000 Tickers by Volume...")
ticker_counts = df['ticker'].value_counts()
all_tickers = ticker_counts.index.tolist()

# CAP AT 2000
TARGET_LIMIT = 2000
tickers_to_run = all_tickers[:TARGET_LIMIT]
print(f"Study limited to {len(tickers_to_run)} tickers.")

# Check for existing results to resume if it crashes
results_file = output_dir / "05_Full_Scale_DL_Results.csv"
if results_file.exists():
    existing_df = pd.read_csv(results_file)
    completed_tickers = existing_df['Ticker'].unique().tolist()
    # Filter out completed ones from our target list
    tickers_to_run = [t for t in tickers_to_run if t not in completed_tickers]
    print(f"Resuming... {len(completed_tickers)} done, {len(tickers_to_run)} remaining.")
else:
    print("Starting fresh run.")

# Configuration
BATCH_SIZE = 50 
batch_results = []

print(f"Starting comparison: 4 Models x 3 Sentiment Variants x {len(tickers_to_run)} Tickers")

for i, ticker in enumerate(tqdm(tickers_to_run, desc="Processing Tickers")):
    ticker_df = df[df['ticker'] == ticker].copy()
    if len(ticker_df) < 100: continue 
    
    # Train/Test Split
    train_size = int(len(ticker_df) * 0.8)
    
    # Iterate through Sentiment Variants
    for source_name, cols in feature_variants.items():
        try:
            X, y = create_sequences(ticker_df, cols)
            X_train, X_test = X[:train_size], X[train_size:]
            y_train, y_test = y[:train_size], y[train_size:]
            
            if len(X_test) < 10: continue

            # Iterate through Deep Learning Models
            for model_type in ['LSTM', 'BiLSTM', 'GRU', 'CNN']:
                
                tf.keras.backend.clear_session()
                
                model = build_model(model_type, (X_train.shape[1], X_train.shape[2]))
                
                early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)
                
                model.fit(
                    X_train, y_train,
                    epochs=15, 
                    batch_size=64, 
                    validation_split=0.1,
                    callbacks=[early_stop],
                    verbose=0
                )
                
                preds = model.predict(X_test, verbose=0).flatten()
                rmse = np.sqrt(mean_squared_error(y_test, preds))
                
                true_dir = np.sign(y_test)
                pred_dir = np.sign(preds)
                acc = np.mean(true_dir == pred_dir) * 100
                
                batch_results.append({
                    "Ticker": ticker,
                    "Model": model_type,
                    "Source": source_name,
                    "RMSE": rmse,
                    "Accuracy": acc,
                    "Timestamp": datetime.now().strftime("%Y-%m-%d %H:%M")
                })
                
        except Exception as e:
            continue

    # --- SAVE BATCH ---
    if (i + 1) % BATCH_SIZE == 0 or (i + 1) == len(tickers_to_run):
        new_df = pd.DataFrame(batch_results)
        
        # Append to CSV
        new_df.to_csv(results_file, mode='a', header=not results_file.exists(), index=False)
        
        # Upload to DB
        try:
            new_df.columns = [c.lower() for c in new_df.columns]
            new_df.to_sql('results_dl_source_ablation', con=db_engine, if_exists='append', index=False)
        except: pass
        
        print(f"Saved batch of {len(batch_results)} results.")
        batch_results = []

print("DL ABLATION (2000 TICKERS) COMPLETE.")